# Municipality Reference Table

This notebook builds the municipality reference table by integrating all prepared municipality-level datasets into a single dataset.

## Pipeline

1. Load prepared datasets
2. Explore input datasets
3. Build municipality table
4. Explore municipality table
5. Validate municipality table
6. Save municipality table

## Input datasets

- Population
- Employment
- Workplace employment
- Municipality boundaries
- Topography

## Output

- `municipality.parquet`

## Notes

The municipality table is the central reference dataset used throughout HERMES.

Each row represents a single French municipality identified by its INSEE code.

This table intentionally excludes:

- mobility flows (origin–destination relationships),
- climate time series (grid × month observations),

which are stored in dedicated datasets because they have different granularities.

In [1]:
# ============================================================================
# Imports
# ============================================================================


from hermes.config import PREPARED_DIR
from hermes.io import (load_dataframe, save_dataframe)

from hermes.loaders import (
    load_population,
    load_employment,
    load_workplace_employment,
    load_topography,
    load_municipality_boundaries,
)

from hermes.integration.municipality import (
    build_municipality_table,
)

from hermes.validation import (
    explore_dataframe,
    validate_dataframe,
)

from hermes.integration.coverage import (
    compare_dataset_coverage,
)

In [2]:
# ============================================================================
# Load datasets
# ============================================================================

population = load_dataframe(
    PREPARED_DIR / "population.parquet"
)

employment = load_dataframe(
    PREPARED_DIR / "employment.parquet"
)

workplace_employment = load_dataframe(
    PREPARED_DIR / "workplace_employment.parquet"
)

municipality_boundaries = load_dataframe(
    PREPARED_DIR / "municipality_boundaries.parquet"
)

topography = load_dataframe(
    PREPARED_DIR / "topography.parquet"
)

INFO | hermes.io | Loading dataset: /Users/rinarazafimahefa/Documents/HERMES/data/prepared/population.parquet
INFO | hermes.io | Loading dataset: /Users/rinarazafimahefa/Documents/HERMES/data/prepared/employment.parquet
INFO | hermes.io | Loading dataset: /Users/rinarazafimahefa/Documents/HERMES/data/prepared/workplace_employment.parquet
INFO | hermes.io | Loading dataset: /Users/rinarazafimahefa/Documents/HERMES/data/prepared/municipality_boundaries.parquet
INFO | hermes.io | Loading dataset: /Users/rinarazafimahefa/Documents/HERMES/data/prepared/topography.parquet


In [3]:
for name, df in {
    "Population": population,
    "Employment": employment,
    "Workplace": workplace_employment,
    "Topography": topography,
}.items():
    print(name, df.shape)

Population (34920, 2)
Employment (34920, 10)
Workplace (34875, 6)
Topography (34868, 7)


In [4]:
# Compare municipality coverage

missing_in_population = (
    set(topography["insee_code"])
    - set(population["insee_code"])
)

len(missing_in_population), sorted(missing_in_population)

(0, [])

In [5]:
missing_in_topography = (
    set(population["insee_code"])
    - set(topography["insee_code"])
)

len(missing_in_topography), sorted(missing_in_topography)

(52,
 ['13201',
  '13202',
  '13203',
  '13204',
  '13205',
  '13206',
  '13207',
  '13208',
  '13209',
  '13210',
  '13211',
  '13212',
  '13213',
  '13214',
  '13215',
  '13216',
  '15031',
  '15035',
  '15047',
  '15141',
  '60054',
  '69381',
  '69382',
  '69383',
  '69384',
  '69385',
  '69386',
  '69387',
  '69388',
  '69389',
  '75101',
  '75102',
  '75103',
  '75104',
  '75105',
  '75106',
  '75107',
  '75108',
  '75109',
  '75110',
  '75111',
  '75112',
  '75113',
  '75114',
  '75115',
  '75116',
  '75117',
  '75118',
  '75119',
  '75120',
  '85084',
  '85165'])

In [6]:
municipality_boundaries.columns.tolist()

['insee_code',
 'municipality',
 'entity_type',
 'parent_insee_code',
 'department_code',
 'region_code',
 'geometry']

In [7]:
municipality_boundaries[
    municipality_boundaries["insee_code"].isin(
        missing_in_population
    )
][
    [
        "insee_code",
        "municipality",
    ]
]

,insee_code,municipality


In [8]:
municipality_boundaries[
    municipality_boundaries["insee_code"].isin(
        {
            "15031",
            "15035",
            "15047",
            "15141",
            "60054",
            "85084",
            "85165",
        }
    )
]

,insee_code,municipality,entity_type,parent_insee_code,department_code,region_code,geometry
5023,15031,Celles,COMMUNE,<NA>,15,84,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
5030,15035,Chalinargues,COMMUNE,<NA>,15,84,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
5043,15047,Chavagnac,COMMUNE,<NA>,15,84,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
5139,15141,Neussargues-Moissac,COMMUNE,<NA>,15,84,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
22600,60054,Beaumont-les-Nonains,COMMUNE,<NA>,60,32,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
32706,85084,Essarts-en-Bocage,COMMUNE,<NA>,85,52,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
32790,85165,L'Oie,COMMUNE,<NA>,85,52,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...


In [9]:
# ============================================================================
# Compare dataset coverage
# ============================================================================

compare_dataset_coverage(
    Population=population,
    Employment=employment,
    Workplace=workplace_employment,
    Municipality_Boundaries=municipality_boundaries,
    Topography=topography,
)

INFO | hermes.integration.coverage | ================================================================================
INFO | hermes.integration.coverage | Dataset coverage
INFO | hermes.integration.coverage | ================================================================================
INFO | hermes.integration.coverage | Population                 34,920 municipalities
INFO | hermes.integration.coverage | Employment                 34,920 municipalities
INFO | hermes.integration.coverage | Workplace                  34,875 municipalities
INFO | hermes.integration.coverage | Municipality_Boundaries    34,922 municipalities
INFO | hermes.integration.coverage | Topography                 34,868 municipalities
INFO | hermes.integration.coverage | 
INFO | hermes.integration.coverage | Reference dataset: Population
INFO | hermes.integration.coverage | Employment: identical coverage
WARNING | hermes.integration.coverage | Workplace: missing 45 municipalities from Population
WARNING | herm

<span style="color:green">
<b>Note on dataset coverage:</b>

The municipality datasets used by HERMES do not all share the same administrative coverage.

Observed differences are expected and originate from the official data sources:

- **Mayotte (INSEE codes `976xx`)** is present in the 2026 topography and municipality boundary datasets but absent from the INSEE 2023 census datasets.
- A small number of municipalities differ because of **administrative changes** (municipality mergers, dissolutions or code changes) between the source vintages.

The municipality reference table is built using the **population dataset as the reference**, ensuring consistent municipality coverage across all integrated indicators.
</span>

In [10]:
# ============================================================================
# Build municipality table
# ============================================================================

municipality = build_municipality_table(
    population,
    employment,
    workplace_employment,
    municipality_boundaries,
    topography,
)

In [11]:
municipality.shape

(34920, 28)

In [12]:
len(municipality["insee_code"].unique())

34920

In [13]:
# ============================================================================
# Explore municipality table
# ============================================================================

explore_dataframe(
    municipality,
    title="Municipality Table",
)

INFO | hermes.validation | ================================================================================
INFO | hermes.validation | Municipality Table
INFO | hermes.validation | ================================================================================
INFO | hermes.validation | Rows: 34,920
INFO | hermes.validation | Columns: 28
INFO | hermes.validation | Memory: 434.51 MB
INFO | hermes.validation | Column summary


,Column,Type,Non-null count,Missing
insee_code,insee_code,object,34920,0
population,population,float64,34920,0
population_15_64,population_15_64,float64,34920,0
active_population,active_population,float64,34920,0
employed,employed,float64,34920,0
unemployed,unemployed,float64,34920,0
inactive,inactive,float64,34920,0
students,students,float64,34903,17
retired,retired,float64,34903,17
homemakers,homemakers,float64,34903,17


INFO | hermes.validation | Numeric summary


,population,population_15_64,active_population,employed,unemployed,inactive,students,retired,homemakers,other_inactive,...,salaried_jobs,self_employed_jobs,full_time_jobs,part_time_jobs,area_hectares,mean_elevation,min_elevation,max_elevation,latitude,longitude
count,3.492000e+04,3.492000e+04,3.492000e+04,3.492000e+04,34920.000000,34920.000000,34903.000000,34903.000000,34903.000000,34903.000000,...,3.487500e+04,34875.000000,3.487500e+04,34875.000000,34739.000000,34863.000000,34863.000000,34863.000000,34863.000000,34863.000000
mean,2.057839e+03,1.277262e+03,9.642141e+02,8.535967e+02,110.617403,313.047955,68.481483,131.918033,35.176483,75.605562,...,6.952546e+02,113.050466,6.820502e+02,126.254840,1579.956965,279.136793,188.654017,384.898804,46.786457,2.715337
std,1.533434e+04,1.044646e+04,7.942704e+03,7.016772e+03,953.347181,2559.112928,298.597734,1420.918108,287.387596,621.687060,...,9.677129e+03,1547.913363,9.591353e+03,1626.742169,1735.908564,288.600684,201.523197,453.071485,3.590414,4.414245
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000e+00,0.000000,3.000000,0.000000,0.000000,0.000000,-21.340000,-61.780000
25%,1.980000e+02,1.159605e+02,9.051351e+01,8.261640e+01,7.000000,24.477742,8.914770,7.164710,2.000000,4.742195,...,1.198832e+01,10.173315,1.898745e+01,4.251195,652.000000,105.000000,55.000000,132.000000,45.068000,0.799500
50%,4.600000e+02,2.743616e+02,2.154721e+02,1.971933e+02,16.903150,58.101605,20.595870,19.215060,5.005060,11.066380,...,4.081419e+01,22.420610,5.192393e+01,12.502160,1098.000000,189.000000,134.000000,231.000000,47.354000,2.720000
75%,1.180000e+03,7.023286e+02,5.521488e+02,5.070000e+02,44.000000,149.382575,51.601205,53.000000,13.554120,30.000000,...,1.843214e+02,58.927540,2.008697e+02,42.289555,1901.500000,337.000000,252.000000,436.000000,48.824000,4.918500
max,2.103778e+06,1.463510e+06,1.152923e+06,1.032139e+06,120784.007570,310586.137270,32286.975870,186207.981650,28468.722080,65189.254330,...,1.551134e+06,260121.109240,1.557444e+06,253811.098870,75764.000000,2713.000000,9589.000000,4808.000000,51.073000,55.754000


INFO | hermes.validation | Categorical summary


,insee_code,municipality,entity_type,geometry
count,34920,34920,34920,34920
unique,34920,32690,2,34920
top,13201,Sainte-Colombe,COMMUNE,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
freq,1,12,34875,1


INFO | hermes.validation | First 5 rows


,insee_code,population,population_15_64,active_population,employed,unemployed,inactive,students,retired,homemakers,...,parent_insee_code,department_code,region_code,geometry,area_hectares,mean_elevation,min_elevation,max_elevation,latitude,longitude
0,13201,37599.0,27153.79405,16947.15220,13892.19213,3054.96007,10206.64185,797.37659,4408.22662,1346.88943,...,13055,13,93,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,NaN,NaN,NaN,NaN,NaN,NaN
1,13202,24378.0,16203.57258,11028.93271,9360.95823,1667.97448,5174.63987,553.85211,1801.87747,685.02342,...,13055,13,93,b'\x01\x06\x00\x00\x00\x02\x00\x00\x00\x01\x03...,NaN,NaN,NaN,NaN,NaN,NaN
2,13203,58029.0,37772.64481,20889.23727,17486.20464,3403.03263,16883.40754,1103.30697,5218.95119,4058.83619,...,13055,13,93,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,NaN,NaN,NaN,NaN,NaN,NaN
3,13204,49363.0,31301.73863,23383.36512,20122.92522,3260.43991,7918.37351,1204.44593,3031.87060,1185.33998,...,13055,13,93,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,NaN,NaN,NaN,NaN,NaN,NaN
4,13205,45702.0,32402.54365,23704.80770,20310.08899,3394.71871,8697.73595,720.25411,5401.70641,524.56185,...,13055,13,93,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Missing values

(
    municipality
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Missing")
)

,Missing
parent_insee_code,34875
area_hectares,181
longitude,57
latitude,57
max_elevation,57
min_elevation,57
mean_elevation,57
total_jobs,45
part_time_jobs,45
self_employed_jobs,45


<span style="color:green">
<b>Note on missing values:</b>

The municipality table inherits the expected missing values from the topography dataset.

These missing values are present in the original source data and are not introduced during the integration process.

Affected variables are:

- `area_hectares`
- `mean_elevation`
- `min_elevation`
- `max_elevation`
- `latitude`
- `longitude`
</span>

In [15]:
# ============================================================================
# Validate municipality table
# ============================================================================

validate_dataframe(
    municipality,
    key="insee_code",
    expected_missing_columns=[
        "area_hectares",
        "mean_elevation",
        "min_elevation",
        "max_elevation",
        "latitude",
        "longitude",
    ],
)

INFO | hermes.validation | ================================================================================
INFO | hermes.validation | Dataset validation
INFO | hermes.validation | ================================================================================
INFO | hermes.validation | Duplicate rows: 0
WARNING | hermes.validation | Missing values: 35634
ERROR | hermes.validation | ✗ students: 17 missing values (unexpected)
ERROR | hermes.validation | ✗ retired: 17 missing values (unexpected)
ERROR | hermes.validation | ✗ homemakers: 17 missing values (unexpected)
ERROR | hermes.validation | ✗ other_inactive: 17 missing values (unexpected)
ERROR | hermes.validation | ✗ total_jobs: 45 missing values (unexpected)
ERROR | hermes.validation | ✗ salaried_jobs: 45 missing values (unexpected)
ERROR | hermes.validation | ✗ self_employed_jobs: 45 missing values (unexpected)
ERROR | hermes.validation | ✗ full_time_jobs: 45 missing values (unexpected)
ERROR | hermes.validation | ✗ part_time_job

In [16]:
# ============================================================================
# Dataset dimensions
# ============================================================================

print(f"Rows    : {len(municipality):,}")
print(f"Columns : {municipality.shape[1]}")

Rows    : 34,920
Columns : 28


In [17]:
# ============================================================================
# Save municipality table
# ============================================================================

save_dataframe(
    municipality,
    PREPARED_DIR / "municipality.parquet",
)

WARNING | hermes.io | Overwriting existing file: /Users/rinarazafimahefa/Documents/HERMES/data/prepared/municipality.parquet
INFO | hermes.io | Dataset saved: /Users/rinarazafimahefa/Documents/HERMES/data/prepared/municipality.parquet


PosixPath('/Users/rinarazafimahefa/Documents/HERMES/data/prepared/municipality.parquet')

In [18]:
municipality[
    municipality["insee_code"].isin(
        ["69123", "69382", "69383", "75056", "75108", "13055", "13212"]
    )
][
    [
        "insee_code",
        "municipality",
    ]
]

,insee_code,municipality
11,13212,Marseille 12e Arrondissement
17,69382,Lyon 2e Arrondissement
18,69383,Lyon 3e Arrondissement
32,75108,Paris 8e Arrondissement
4383,13055,Marseille
27084,69123,Lyon
29239,75056,Paris


In [19]:
print(municipality.columns.tolist())

['insee_code', 'population', 'population_15_64', 'active_population', 'employed', 'unemployed', 'inactive', 'students', 'retired', 'homemakers', 'other_inactive', 'total_jobs', 'salaried_jobs', 'self_employed_jobs', 'full_time_jobs', 'part_time_jobs', 'municipality', 'entity_type', 'parent_insee_code', 'department_code', 'region_code', 'geometry', 'area_hectares', 'mean_elevation', 'min_elevation', 'max_elevation', 'latitude', 'longitude']
